## Week 4 Homework: Evaluation

In this notebook we generate a ground truth dataset and use it to evaluate
search, the same way we did in the module. There we only evaluated keyword
search. Here we also evaluate vector and hybrid search, so we can finally
compare them on numbers instead of intuition.



## Setup

run this on the terminal: 
```
uv add openai pydantic python-dotenv pandas onnxruntime tokenizers numpy tqdm minsearch

```

run this to download the onnx model `all-MiniLM-L6-v2` for the vector search & embeddings

In [1]:
# Load the same data as in week 2's homework
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
len(documents)

72

This gives 72 pages.

## Generating ground truth

To evaluate search, we need a dataset of questions where we know which document
is the correct answer. This is the ground truth.

We generate it the same way as in the module. For each lesson page, we ask an
LLM to write 5 questions that are answered by that page. Each question is then
labeled with the page it came from.

We use the same structured-output approach as in the module - the same
`Questions` model and the `llm_structured` helper from `evaluation_utils.py`.

Download `evaluation_utils.py` and the `rag_helper.py` it depends on:

# run this in terminal: 
```
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
wget ${PREFIX}/04-evaluation/code/evaluation_utils.py
```

The module's instructions generate questions from a FAQ record, so we adapt
them for a lesson page:

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

We ask for different wording from the page on purpose. Real users don't phrase
their questions the way the lesson does, and copying the text would make the
evaluation too easy.

For each page, build a JSON user prompt from its `filename` and `content`, then
call `llm_structured` with the `Questions` model. Turn each returned question
into a record labeled with the page's `filename`. The call also returns the
token usage, the same as in the lessons.

## Q1. Generating questions

Generating questions for all 72 pages costs money and takes time, so let's
start small and generate questions for just the first 3 pages:

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/02-environment.md`
- `01-agentic-rag/lessons/03-rag.md`

Each call returns the token usage, which most LLM APIs report on the response
object (e.g. `response.usage.input_tokens` / `prompt_tokens`).

What's the average number of input tokens across these 3 calls?

* [] 140
* [x] 1400
* [] 14000
* [] 140000

> These numbers vary between runs, even with the same model, so pick the closest
> option. A different provider or model may land further apart, but the input
> tokens stay in the same order of magnitude - the prompt we send is the same.

In [3]:
from pydantic import BaseModel


class Questions(BaseModel):
    questions: list[str]

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
import json
from evaluation_utils import llm_structured_retry


def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client, data_gen_instructions, user_prompt, Questions
    )

    results = []

    for q in out.questions:
        results.append({"question": q, "document": doc["filename"]})

    return results, usage

In [6]:
# try creating synthetic ground truth for first 5 docs
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [7]:
q1_answer = 0

for usage in usages:
    q1_answer += usage.input_tokens

print(q1_answer / 3)


1353.0


## The full ground truth

You don't need to generate the data for the rest of the homework. We already
did it for all 72 pages, using the same approach as in the lessons, and saved
the 360 questions to a file.

Run this in terminal:
```
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv
```

Load it with pandas into a list of records called `ground_truth`. Each record
has a `question` and the `filename` of the page that should answer it.

In [8]:
import pandas as pd

ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")

## Searching the chunks

We search over the same chunks as in homework 2.

Create them with `chunk_documents`:

In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


In [10]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [11]:
# one-time set up for text search
from minsearch import Index, VectorSearch

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

In [12]:
# one-time set up for vector search
from embedder import Embedder
import numpy as np

embed = Embedder()
vectors = []


for chunk in chunks:
    chunk_vector = embed.encode(chunk["content"])
    vectors.append(chunk_vector)
X = np.array(vectors)

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

2026-07-09 14:59:06.650250306 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Set up embedder, and fit vector 

This gives 295 chunks.

Now rebuild the search from homework 2 over these chunks. Build a text index
(`Index`) and a vector index (`VectorSearch`), both keyed on `filename`. Wrap
each one in a function, `text_search` and `vector_search`, that takes a query
and the number of results to return (5 by default).

For hybrid search, reuse the `rrf` function from homework 2:

In [34]:
def text_search(query, num_results=5, raw=False):
    text_search_results = text_index.search(query, num_results=num_results)
    if raw:
        return text_search_results  # used for hybrid_search() because rrf needs access to raw dictionary
    else:
        text_results_list = [t["filename"] for t in text_search_results]
        return text_results_list


def vector_search(query, num_results=5, raw=False):
    query_vector = embed.encode(query)
    vector_search_results = vindex.search(query_vector, num_results=num_results)
    if raw:
        return vector_search_results
    else:
        vector_search_results_list = [t["filename"] for t in vector_search_results]
        return vector_search_results_list


In [35]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Then define `hybrid_search` on top of it:

In [ ]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10, raw=True)
    vector_results = vector_search(query, num_results=10, raw=True)
    return rrf([text_results, vector_results], k=k)

## Q2. First result with text search

Take the first question from the ground truth:

In [37]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

After running `text_search` for it, what's the `filename` of the first result?

* [] `01-agentic-rag/lessons/01-intro.md`
* [x] `01-agentic-rag/lessons/03-rag.md`
* [] `01-agentic-rag/lessons/13-function-calling.md`
* [] `01-agentic-rag/lessons/10-rag-next-steps.md`

In [38]:
results = text_search(q)
results

['01-agentic-rag/lessons/03-rag.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/03-rag.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/01-intro.md']

## Q3. First result with vector search

After running `vector_search` for the same question, what's the `filename` of
the first result?

* [x] `01-agentic-rag/lessons/01-intro.md`
* [] `01-agentic-rag/lessons/03-rag.md`
* [] `04-evaluation/lessons/11-evaluation-intro.md`
* [] `04-evaluation/lessons/12-rag-answers.md`

This question was generated from `01-agentic-rag/lessons/01-intro.md`. Notice
that one method finds the right page at the top and the other doesn't. That's
exactly why we measure across the whole dataset instead of trusting one query.

In [ ]:
results = vector_search(q)
results

['01-agentic-rag/lessons/01-intro.md',
 '04-evaluation/lessons/11-evaluation-intro.md',
 '04-evaluation/lessons/12-rag-answers.md',
 '01-agentic-rag/lessons/10-rag-next-steps.md',
 '06-best-practices/lessons/01-intro.md']

## Evaluation metrics

We evaluate search exactly as in the module, reusing the same functions from the
lecture. We change only the label. Our ground truth uses `filename`, so a result
counts as a hit when a returned chunk's `filename` matches the question's
`filename`, not a document `id`.

As a reminder, these functions do the following:

- `compute_relevance` runs search for a question and returns a list of 0s and 1s
- `hit_rate` is the fraction of questions where the correct page appears in the results
- `mrr` (Mean Reciprocal Rank) also rewards finding the page near the top
- `evaluate` runs a search function over the whole ground truth and returns both metrics

## Q4. Evaluating text search

Evaluate `text_search` on the ground truth data.

What's the Hit Rate?

* [] 0.55
* [] 0.66
* [x] 0.76
* [] 0.88

In [40]:
ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [41]:
def compute_relevance_text(ground_truth):
    filename = ground_truth["filename"]
    results = text_search(query=ground_truth["question"])

    relevance = []
    for d in results:
        relevance.append(int(d == filename))

    return relevance


def compute_relevance(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)
    return relevance_total


relevance_total = compute_relevance(ground_truth=ground_truth)


  0%|          | 0/360 [00:00<?, ?it/s]

In [42]:
relevance_total[:5]


[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0]]

In [43]:
def hit_rate(relevance_total):
    hit_rate = 0
    for row in relevance_total:
        if any(row) == 1:
            hit_rate += 1
            pass  # if any of the rows is 1, then count as 1
    return hit_rate / len(relevance_total)


hr = hit_rate(relevance_total)
hr

0.7583333333333333

## Q5. Evaluating vector search

Now evaluate `vector_search` - the part we left for the homework, since the
module only evaluated keyword search.

What's the MRR?

* [] 0.35
* [] 0.45
* [x] 0.55
* [] 0.65

In [44]:
def compute_relevance_vector(ground_truth):
    filename = ground_truth["filename"]
    results = vector_search(query=ground_truth["question"])

    relevance = []
    for d in results:
        relevance.append(int(d == filename))

    return relevance


def compute_relevance_total_vector(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_vector(q)
        relevance_total.append(relevance)
    return relevance_total


def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)


In [24]:
relevance_vector_total = compute_relevance_total_vector(ground_truth)

  0%|          | 0/360 [00:00<?, ?it/s]

In [25]:
mrr_result = mrr(relevance_vector_total)
print(mrr_result)


0.5486111111111112


## Q6. Tuning hybrid search

The `k` constant in RRF controls how much the top ranks matter. A smaller `k`
sharpens the gap between positions, so being at the top of a list counts for
more. The RRF paper uses 60 as a default, but the best value depends on the data
- so let's measure it.

Evaluate `hybrid_search` over the full ground truth dataset for `k` values 1,
50, 100, and 200. Compare the MRR values for these runs.

Which `k` gives the best MRR?

* [x] 1
* [] 50
* [] 100
* [] 200

> Several values of `k` may give the same MRR. If there's a tie, pick the
> smallest `k`.

In [46]:
def compute_relevance_hybrid(ground_truth, k):
    filename = ground_truth["filename"]
    results = hybrid_search(ground_truth["question"], k=k)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance


def compute_relevance_total_hybrid(ground_truth, k):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_hybrid(q, k=k)
        relevance_total.append(relevance)
    return relevance_total


for k in [1, 50, 100, 200]:
    relevance_total = compute_relevance_total_hybrid(ground_truth, k=k)
    mrr_results = mrr(relevance_total)
    print(f"k={k}: {mrr_results}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: 0.6481944444444449


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: 0.637916666666667


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: 0.637916666666667


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: 0.637916666666667


## Using this framework

You now have an `evaluate` function that takes any search function and returns
Hit Rate and MRR.

Use it to measure any change you make to search:

- tune the field boosts in keyword search
- try a different embedding model for vector search
- change `k` in the RRF formula for hybrid search
- change the number of results you return

Change a setting, re-run `evaluate`, and see whether the metric moves. The
ground truth stays fixed, so the comparison is fair. That's how you replace
guessing with measuring.

## Submit the results

* Submit your results here: https://courses.datatalks.club/llm-zoomcamp-2026/homework/hw4
* It's possible your answers won't match exactly. If so, select the closest one.